# Splitting the Matmul: Tensor Parallelism from Scratch

When a model outgrows one GPU, tensor parallelism (TP) splits every weight matrix
across devices and keeps all of them busy on the *same* token — the Megatron-LM recipe
([Shoeybi et al., 2019](https://arxiv.org/abs/1909.08053)) behind every `tp_size=8`
flag you've ever passed.

This notebook builds it from nothing, on a deliberately *small* GPT so the whole thing
runs in seconds on 2 GPUs — and so it can demonstrate the result most tutorials hide:

> **At small scale, TP makes things *slower*.** Our 29M-parameter model drops to 0.62×
> single-GPU speed on 2×H200, because the all-reduce costs more than the matmuls it
> parallelizes. TP is a *memory* lever that only becomes a *speed* lever once per-layer
> compute dwarfs interconnect latency. We measure exactly where that crossover lives.

Plan: baseline GPT → column/row-parallel layers with their communication rules →
2-GPU benchmark (memory ↓, speed ↓!) → shard inspection → all-reduce microbenchmark →
the compute-vs-communication crossover sweep.

**Requires:** 2+ CUDA GPUs and `torchrun`.

## 0 · Environment

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader
import torch
print(f"torch {torch.__version__} · {torch.cuda.device_count()} GPU(s)")

## 1 · The reference model

A plain GPT: `d_model 512 · 8 heads · d_ff 2048 · 6 layers · vocab 10k` — about 29M
parameters. Small enough to iterate in seconds, structured exactly like the real thing.
Written to a module file so the single-GPU and torchrun scripts share one definition.

In [ ]:
%%writefile tp_common.py
"""Shared config + reference (single-GPU) model for the TP tutorial."""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

CFG = dict(d_model=512, n_heads=8, d_ff=2048, n_layers=6,
           vocab=10_000, max_seq=512, batch=8, seq=256,
           warmup=3, iters=10)


class Attention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.h, self.d = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, S, E = x.shape
        q, k, v = self.qkv(x).view(B, S, 3, self.h, self.d).permute(2, 0, 3, 1, 4).unbind(0)
        att = q @ k.transpose(-1, -2) / math.sqrt(self.d)
        att = att.masked_fill(torch.ones(S, S, device=x.device, dtype=torch.bool).triu(1),
                              float("-inf"))
        return self.proj((att.softmax(-1) @ v).transpose(1, 2).reshape(B, S, E))


class Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = Attention(d_model, n_heads)
        self.mlp = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(),
                                 nn.Linear(d_ff, d_model))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class MiniGPT(nn.Module):
    def __init__(self, cfg=CFG):
        super().__init__()
        self.tok = nn.Embedding(cfg["vocab"], cfg["d_model"])
        self.pos = nn.Embedding(cfg["max_seq"], cfg["d_model"])
        self.blocks = nn.ModuleList(Block(cfg["d_model"], cfg["n_heads"], cfg["d_ff"])
                                    for _ in range(cfg["n_layers"]))
        self.norm = nn.LayerNorm(cfg["d_model"])
        self.head = nn.Linear(cfg["d_model"], cfg["vocab"], bias=False)

    def forward(self, ids):
        x = self.tok(ids) + self.pos(torch.arange(ids.shape[1], device=ids.device))
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.norm(x))


def param_count(m):
    return sum(p.numel() for p in m.parameters())

## 2 · Baseline: one GPU

Full training step (forward + backward + Adam) so the numbers include gradients and
optimizer state — the memory that actually forces sharding in practice.

In [ ]:
%%writefile bench_single.py
import json
import time

import torch
import torch.nn.functional as F

from tp_common import CFG, MiniGPT, param_count


def timed_steps(model, opt, ids, labels, n):
    times = []
    for _ in range(n):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        loss = F.cross_entropy(model(ids).flatten(0, 1), labels.flatten())
        loss.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return sum(times) / len(times), loss.item()


def main():
    torch.manual_seed(0)
    model = MiniGPT().cuda()
    opt = torch.optim.Adam(model.parameters(), lr=3e-4)
    weights_mb = torch.cuda.memory_allocated() / 2**20

    ids = torch.randint(0, CFG["vocab"], (CFG["batch"], CFG["seq"]), device="cuda")
    labels = torch.randint(0, CFG["vocab"], (CFG["batch"], CFG["seq"]), device="cuda")

    timed_steps(model, opt, ids, labels, CFG["warmup"])
    torch.cuda.reset_peak_memory_stats()
    step_s, loss = timed_steps(model, opt, ids, labels, CFG["iters"])

    out = dict(gpus=1, params_per_gpu=param_count(model), weights_mb=round(weights_mb, 1),
               peak_mb=round(torch.cuda.max_memory_allocated() / 2**20, 1),
               step_ms=round(step_s * 1000, 2),
               tok_s=round(CFG["batch"] * CFG["seq"] / step_s), loss=round(loss, 4))
    json.dump(out, open("single_gpu.json", "w"), indent=2)
    print(json.dumps(out, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
!python bench_single.py

## 3 · The two sharded layers

Megatron's insight: a transformer block is two matmul sandwiches, and each can be split
so that only **one all-reduce per sandwich** is needed.

**Column-parallel** — split the weight's *output* dimension. Every rank holds
`W[:, rank-slice]`, sees the full input, and produces its own slice of the output.
No communication in forward; the backward must all-reduce input gradients.

**Row-parallel** — split the *input* dimension. Every rank holds `W[rank-slice, :]`,
consumes the sliced activation the column layer produced, and computes a *partial* sum
of the full output. Forward must **all-reduce**; backward is communication-free.

Composed as column → (nonlinearity) → row, the sliced activations never leave the rank:

```
attention:  qkv = column (heads split across ranks)     out-proj = row  → all-reduce
mlp:        up  = column (d_ff split)                   down     = row  → all-reduce
```

Two all-reduces per block, total. The communication rules live in two tiny autograd
functions — identity one way, all-reduce the other — which is the entire "magic" of TP.

## 4 · The TP model and benchmark

Same model, same step, but every block is sharded. `torchrun` runs one process per GPU;
NCCL provides the all-reduces.

In [ ]:
%%writefile bench_tp.py
import json
import math
import time

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F

from tp_common import CFG, param_count


class _IdentityFwdAllreduceBwd(torch.autograd.Function):
    """Placed before a column-parallel layer."""
    @staticmethod
    def forward(ctx, x):
        return x

    @staticmethod
    def backward(ctx, g):
        dist.all_reduce(g)
        return g


class _AllreduceFwdIdentityBwd(torch.autograd.Function):
    """Placed after a row-parallel layer."""
    @staticmethod
    def forward(ctx, x):
        dist.all_reduce(x)
        return x

    @staticmethod
    def backward(ctx, g):
        return g


class ColumnSharded(nn.Module):
    """out_features split across ranks; forward needs no communication."""

    def __init__(self, d_in, d_out):
        super().__init__()
        ws = dist.get_world_size()
        assert d_out % ws == 0
        self.weight = nn.Parameter(torch.empty(d_out // ws, d_in))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))

    def forward(self, x):
        return F.linear(_IdentityFwdAllreduceBwd.apply(x), self.weight)


class RowSharded(nn.Module):
    """in_features split across ranks; forward all-reduces the partial sums."""

    def __init__(self, d_in, d_out):
        super().__init__()
        ws = dist.get_world_size()
        assert d_in % ws == 0
        self.weight = nn.Parameter(torch.empty(d_out, d_in // ws))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))

    def forward(self, x):
        return _AllreduceFwdIdentityBwd.apply(F.linear(x, self.weight))


class ShardedAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        ws = dist.get_world_size()
        assert n_heads % ws == 0
        self.h_local, self.d = n_heads // ws, d_model // n_heads
        self.qkv = ColumnSharded(d_model, 3 * d_model)
        self.proj = RowSharded(d_model, d_model)

    def forward(self, x):
        B, S, _ = x.shape
        q, k, v = (self.qkv(x).view(B, S, 3, self.h_local, self.d)
                   .permute(2, 0, 3, 1, 4).unbind(0))
        att = q @ k.transpose(-1, -2) / math.sqrt(self.d)
        att = att.masked_fill(torch.ones(S, S, device=x.device, dtype=torch.bool).triu(1),
                              float("-inf"))
        out = (att.softmax(-1) @ v).transpose(1, 2).reshape(B, S, self.h_local * self.d)
        return self.proj(out)                       # <- the block's first all-reduce


class ShardedBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)  # replicated
        self.attn = ShardedAttention(d_model, n_heads)
        self.up = ColumnSharded(d_model, d_ff)
        self.down = RowSharded(d_ff, d_model)       # <- the block's second all-reduce

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.down(F.gelu(self.up(self.norm2(x))))


class ShardedGPT(nn.Module):
    def __init__(self, cfg=CFG):
        super().__init__()
        self.tok = nn.Embedding(cfg["vocab"], cfg["d_model"])   # replicated (small)
        self.pos = nn.Embedding(cfg["max_seq"], cfg["d_model"])
        self.blocks = nn.ModuleList(
            ShardedBlock(cfg["d_model"], cfg["n_heads"], cfg["d_ff"])
            for _ in range(cfg["n_layers"]))
        self.norm = nn.LayerNorm(cfg["d_model"])
        self.head = nn.Linear(cfg["d_model"], cfg["vocab"], bias=False)

    def forward(self, ids):
        x = self.tok(ids) + self.pos(torch.arange(ids.shape[1], device=ids.device))
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.norm(x))


def main():
    dist.init_process_group("nccl")
    rank, ws = dist.get_rank(), dist.get_world_size()
    torch.cuda.set_device(rank)
    torch.manual_seed(0)

    model = ShardedGPT().cuda()
    opt = torch.optim.Adam(model.parameters(), lr=3e-4)
    weights_mb = torch.cuda.memory_allocated() / 2**20

    torch.manual_seed(1)   # same data on every rank
    ids = torch.randint(0, CFG["vocab"], (CFG["batch"], CFG["seq"]), device="cuda")
    labels = torch.randint(0, CFG["vocab"], (CFG["batch"], CFG["seq"]), device="cuda")

    def step():
        loss = F.cross_entropy(model(ids).flatten(0, 1), labels.flatten())
        loss.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)
        return loss

    for _ in range(CFG["warmup"]):
        step()
    torch.cuda.reset_peak_memory_stats()
    dist.barrier()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(CFG["iters"]):
        loss = step()
    torch.cuda.synchronize()
    step_s = (time.perf_counter() - t0) / CFG["iters"]

    if rank == 0:
        out = dict(gpus=ws, params_per_gpu=param_count(model),
                   weights_mb=round(weights_mb, 1),
                   peak_mb=round(torch.cuda.max_memory_allocated() / 2**20, 1),
                   step_ms=round(step_s * 1000, 2),
                   tok_s=round(CFG["batch"] * CFG["seq"] / step_s),
                   loss=round(loss.item(), 4))
        json.dump(out, open("tp_gpus.json", "w"), indent=2)
        print(json.dumps(out, indent=2))
    dist.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
import torch
NPROC = min(torch.cuda.device_count(), 2)
!torchrun --nproc_per_node={NPROC} bench_tp.py

## 5 · Side by side

In [ ]:
import json

single = json.load(open("single_gpu.json"))
tp = json.load(open("tp_gpus.json"))
n = tp["gpus"]
speedup = single["step_ms"] / tp["step_ms"]

print(f"{'metric':<22}{'1 GPU':>14}{f'TP={n}':>14}")
print("-" * 50)
for key, fmt in [("params_per_gpu", ","), ("weights_mb", ".1f"),
                 ("peak_mb", ".1f"), ("step_ms", ".2f"), ("tok_s", ","), ("loss", ".4f")]:
    print(f"{key:<22}{single[key]:>14{fmt}}{tp[key]:>14{fmt}}")
print("-" * 50)
print(f"weights per GPU   : {(1 - tp['params_per_gpu'] / single['params_per_gpu']):.1%} smaller"
      f"  (not 1/{n} — LayerNorms & embeddings replicated)")
print(f"peak memory       : {(1 - tp['peak_mb'] / single['peak_mb']):.1%} smaller"
      f"  (activations at full d_model around every norm/residual)")
print(f"speed             : {speedup:.2f}x vs ideal {n}.00x -> {speedup / n:.0%} efficiency")

## 6 · Where did the weights go?

Print block 0's parameters on each rank: the four matmuls are sharded, the LayerNorms
are replicated — the sharding map every TP implementation shares.

In [ ]:
%%writefile show_shards.py
import torch
import torch.distributed as dist

from bench_tp import ShardedGPT
from tp_common import CFG


def main():
    dist.init_process_group("nccl")
    rank, ws = dist.get_rank(), dist.get_world_size()
    torch.cuda.set_device(rank)
    model = ShardedGPT().cuda()
    if rank == 0:
        print(f"block 0 parameter shapes on rank 0 (TP={ws}):\n")
        for name, p in model.blocks[0].named_parameters():
            kind = ("column-sharded" if any(s in name for s in ("qkv", "up")) else
                    "row-sharded" if any(s in name for s in ("proj", "down")) else
                    "REPLICATED")
            print(f"  {name:<22} {str(list(p.shape)):<14} {kind}")
    dist.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
!torchrun --nproc_per_node={NPROC} show_shards.py

## 7 · The all-reduce bill, and where TP starts paying

Two microbenchmarks:

1. **All-reduce latency vs. message size** — small messages are *latency*-bound: below
   ~1 MB the wire time barely changes, meaning every all-reduce costs ~100 µs no matter
   how little you send. Our model sends 2 MB twelve times per forward.
2. **The crossover sweep** — for growing `d_model`, time one sharded MLP matmul against
   the all-reduce that follows it. TP efficiency is `compute / (compute + comm)`; the
   sweep shows it climbing as matmuls grow into the interconnect latency.

In [ ]:
%%writefile comm_sweep.py
import time

import torch
import torch.distributed as dist


def timeit(fn, iters=20):
    for _ in range(5):
        fn()
    torch.cuda.synchronize()
    dist.barrier()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1e6      # microseconds


def main():
    dist.init_process_group("nccl")
    rank, ws = dist.get_rank(), dist.get_world_size()
    torch.cuda.set_device(rank)

    if rank == 0:
        print(f"\nall-reduce latency vs message size (TP={ws}, bf16):")
        print(f"  {'elements':>12} {'MB':>8} {'us':>10}")
    for n in (1024, 65_536, 1_048_576, 16_777_216, 134_217_728):
        x = torch.randn(n, device="cuda", dtype=torch.bfloat16)
        us = timeit(lambda: dist.all_reduce(x))
        if rank == 0:
            print(f"  {n:>12,} {n * 2 / 1e6:>8.2f} {us:>10.1f}")

    tokens = 2048                       # batch x seq rows entering the MLP
    if rank == 0:
        print(f"\ncrossover sweep — sharded MLP matmul vs its all-reduce ({tokens} tokens):")
        print(f"  {'d_model':>8} {'matmul us':>11} {'allreduce us':>13} {'TP efficiency':>14}")
    for d in (512, 1024, 2048, 4096, 8192):
        x = torch.randn(tokens, d, device="cuda", dtype=torch.bfloat16)
        w = torch.randn(d, 4 * d // ws, device="cuda", dtype=torch.bfloat16)
        act = torch.randn(tokens * d, device="cuda", dtype=torch.bfloat16)
        c = timeit(lambda: x @ w)
        a = timeit(lambda: dist.all_reduce(act))
        if rank == 0:
            print(f"  {d:>8} {c:>11.1f} {a:>13.1f} {c / (c + a):>13.0%}")
    dist.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
!torchrun --nproc_per_node={NPROC} comm_sweep.py

## 8 · Reference results (measured, 2× NVIDIA H200 · torch 2.4.1)

The equivalent benchmark on this architecture measured:

| Metric | 1 GPU | TP=2 |
|---|---:|---:|
| Params per GPU | 29.4M | 17.4M (−40.8%) |
| Weight memory | 113.1 MB | 66.8 MB (−40.9%) |
| Peak memory | 1,204 MB | 837 MB (−30.5%) |
| Step time | 11.84 ms | 19.19 ms |
| **Speedup** | 1× | **0.62× — slower!** (31% efficiency) |

And the microbenchmarks explain exactly why:

- all-reduce latency floor ≈ **105–150 µs** even for tiny messages (2 MB activation:
  151 µs; 0.13 MB: 106 µs) — it's latency, not bandwidth;
- at `d_model=512`, the sharded matmul takes **24 µs** but its all-reduce takes
  **46 µs** — communication is ~2× the compute it parallelizes;
- 12 all-reduces per forward × ~2 (backward) ≈ a millisecond of pure latency per step
  on a model whose entire single-GPU step is 12 ms.

**Why the numbers aren't 1/N:** LayerNorms, embeddings, and the LM head are replicated
(params −41%, not −50%); every residual and norm runs at full `d_model` on every rank
(peak memory −31%, not −50%).

**Why real deployments still use TP:** scale the crossover sweep forward. A Llama-70B
layer at `d_model=8192` does ~4,000× more matmul work per all-reduce than this toy —
compute swamps the latency floor, efficiency climbs toward 80–90% on NVLink, and, more
fundamentally, the model *does not fit* on one GPU at all. TP's first job is memory;
speed is what it charges for it.

### References

- Shoeybi et al., *Megatron-LM: Training Multi-Billion Parameter Language Models Using
  Model Parallelism*, 2019 — [arXiv:1909.08053](https://arxiv.org/abs/1909.08053)
- Narayanan et al., *Efficient Large-Scale Language Model Training on GPU Clusters*,
  2021 — [arXiv:2104.04473](https://arxiv.org/abs/2104.04473)
- PyTorch distributed / NCCL — [pytorch.org/docs/stable/distributed.html](https://pytorch.org/docs/stable/distributed.html)
